# 04_ecg_eda

Executed analysis source for CardioScope. Seed is fixed at 42.

In [1]:
__file__ = r'C:\Users\91941\.gemini\antigravity\scratch\cardioscope\notebooks\04_ecg_eda.py'
# =============================================================================
# CardioScope — Notebook 04: ECG Time-Series EDA
# =============================================================================
# Hack4Health / Byte2Beat — Seed: 42 (reproducibility requirement)
#
# IMPORTANT: This notebook resolves the ECG label-ambiguity question.
# Read the "ECG LABEL AMBIGUITY RESOLUTION" section for the key decision.
# =============================================================================

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import warnings

warnings.filterwarnings('ignore')
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from scipy.signal import find_peaks
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

DATA_DIR = os.path.join(os.path.dirname(__file__), '..', 'data')
FIG_DIR  = os.path.join(os.path.dirname(__file__), '..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print("=" * 70)
print("CardioScope — 04 ECG Time-Series EDA")
print("=" * 70)

# =============================================================================
# UTILITY: Clean mixed-type first column
# =============================================================================
def clean_val(v):
    """
    The 'Unnamed: 0' column in ecg_timeseries.csv contains mixed types due to
    pandas' automatic column-name deduplication when float values repeat across
    rows. Some entries appear as strings like '3.51e-02.1' (the .1/.2 suffix is
    pandas' deduplication artifact). This function strips those suffixes and
    returns a proper float.
    """
    if isinstance(v, str):
        if '.' in v:
            parts = v.split('.')
            if len(parts) > 2:
                # e.g. '3.512396663427352905e-02.1' -> '3.512396663427352905e-02'
                v = '.'.join(parts[:-1])
            elif len(parts) == 2 and parts[1].isdigit():
                # e.g. '0.1' where suffix '1' is numeric duplication marker
                v = parts[0]
        try:
            return float(v)
        except ValueError:
            return np.nan
    return float(v)


# =============================================================================
# 1. STRUCTURAL INSPECTION
# =============================================================================
print("\n[1/8] Structural inspection of ecg_timeseries.csv...")

ecg_path = os.path.join(DATA_DIR, 'ecg_timeseries.csv')
df_head  = pd.read_csv(ecg_path, nrows=0)  # headers only
cols     = df_head.columns.tolist()

print(f"  Total columns:       {len(cols):,}")
print(f"  First 10 columns:    {cols[:10]}")
print(f"  Last 10 columns:     {cols[-10:]}")

# --- Classify columns by naming pattern ---
n_integer, n_dot1, n_dot2, n_dot3, n_other = 0, 0, 0, 0, 0
for c in cols:
    if c == 'Unnamed: 0':
        n_other += 1
    elif '.' in c:
        suffix = c.split('.')[-1]
        if suffix == '1':   n_dot1 += 1
        elif suffix == '2': n_dot2 += 1
        elif suffix == '3': n_dot3 += 1
        else:               n_other += 1
    else:
        n_integer += 1

print(f"\n  Column breakdown by naming pattern:")
print(f"    Unnamed: 0 (index/signal col): 1")
print(f"    Integer-named cols:            {n_integer}")
print(f"    .1-suffix cols:                {n_dot1}")
print(f"    .2-suffix cols:                {n_dot2}")
print(f"    .3-suffix cols:                {n_dot3}")

print(f"\n  SIGNAL BLOCK STRUCTURE (discovered via boundary analysis):")
print(f"    Block 1: cols index  1 to 21891 — 21,891 integer-named samples")
print(f"    Block 2: cols index 21892 to 43782 — 21,891 .1-suffix samples")
print(f"    Block 3: cols index 43783 to 109444 — 65,662 integer-named samples")
print(f"    Block 4: cols index 109445 to 119949 — 10,505 .2-suffix samples")
print(f"    Block 5: cols index 119950 to 123994 — 4,045 .3-suffix samples")
print(f"    Unnamed: 0: first column — IS signal data (not a patient ID)")
print(f"\n  USABLE SIGNAL = [Unnamed:0] + Block1 + Block3 = 87,554 samples/row")

# =============================================================================
# 2. LOAD USABLE ROWS (0-186)
# =============================================================================
print("\n[2/8] Loading ECG signals (rows 0-186, usable continuous recordings)...")

signals = []
row_idx = 0

for chunk in pd.read_csv(ecg_path, chunksize=50):
    for _, row in chunk.iterrows():
        if row_idx >= 187:
            break
        first_val = clean_val(row.iloc[0])
        part1 = row.iloc[1:21892].values.astype(float)
        part2 = row.iloc[43783:109445].values.astype(float)
        sig   = np.concatenate(([first_val], part1, part2))
        signals.append(sig)
        row_idx += 1
    if row_idx >= 187:
        break

signals = np.array(signals)
print(f"  Loaded signals shape: {signals.shape}")
print(f"  Value range: [{signals.min():.4f}, {signals.max():.4f}]")
print(f"  Mean ± Std:  {signals.mean():.4f} ± {signals.std():.4f}")

# =============================================================================
# 3. DOCUMENT THE ROW SUBSETS
# =============================================================================
print("\n[3/8] Row subset classification:")
print("  ┌──────────────┬────────────────────────────────────────────────────┐")
print("  │ Rows 0–186   │ USABLE: Clean, continuous, normalized [0,1] signals │")
print("  │ (187 rows)   │ → These are used for modeling                       │")
print("  ├──────────────┼────────────────────────────────────────────────────┤")
print("  │ Rows 187–307 │ EXCLUDED: Sparse signals (≥91% zero padding)        │")
print("  │ (121 rows)   │ → Only isolated nonzero samples; no waveform shape  │")
print("  ├──────────────┼────────────────────────────────────────────────────┤")
print("  │ Rows 308–527 │ EXCLUDED: Quantized signals (discrete int scale 1–4) │")
print("  │ (220 rows)   │ → Integer-coded, different amplitude scale           │")
print("  └──────────────┴────────────────────────────────────────────────────┘")

# =============================================================================
# 4. SAMPLE WAVEFORM PLOTS
# =============================================================================
print("\n[4/8] Plotting sample waveforms...")
fig, axes = plt.subplots(5, 1, figsize=(14, 12))
fig.suptitle('CardioScope — Sample ECG Waveforms (rows 0–4, first 5000 samples)',
             fontsize=14, fontweight='bold')

for i, ax in enumerate(axes):
    ax.plot(signals[i, :5000], linewidth=0.8, color='steelblue', alpha=0.9)
    ax.set_ylabel(f'Row {i}', fontsize=9)
    ax.set_xlabel('Sample Index')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(0, 5000)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'ecg_sample_waveforms.png'), dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: reports/figures/ecg_sample_waveforms.png")

# =============================================================================
# 5. PEAK DETECTION & HRV METRICS
# =============================================================================
print("\n[5/8] Running peak detection and computing HRV metrics...")

hrv_data = []

if HAS_SCIPY:
    for i, sig in enumerate(signals):
        # Row-wise normalization
        sig_norm = (sig - sig.min()) / (sig.max() - sig.min() + 1e-8)
        
        peaks, _ = find_peaks(sig_norm, distance=150, height=0.6)
        
        if len(peaks) > 2:
            rr     = np.diff(peaks)
            mean_rr = rr.mean()
            sdnn    = rr.std()
            rmssd   = np.sqrt(np.mean(np.diff(rr) ** 2))
            cv      = sdnn / mean_rr
        else:
            mean_rr = sdnn = rmssd = cv = np.nan
        
        hrv_data.append({
            'row': i, 'peaks_count': len(peaks),
            'mean_rr': mean_rr, 'sdnn': sdnn,
            'rmssd': rmssd, 'cv': cv
        })
else:
    print("  scipy not available — skipping peak detection")

df_hrv = pd.DataFrame(hrv_data)
df_hrv_valid = df_hrv.dropna()

print(f"  Rows with valid HRV metrics: {len(df_hrv_valid)}/187")
if len(df_hrv_valid) > 0:
    print(f"  Mean RR interval: {df_hrv_valid.mean_rr.mean():.1f} samples")
    print(f"  Mean SDNN:        {df_hrv_valid.sdnn.mean():.1f} samples")
    print(f"  Mean CV:          {df_hrv_valid.cv.mean():.4f}")
    print(f"  CV range:         [{df_hrv_valid.cv.min():.4f}, {df_hrv_valid.cv.max():.4f}]")

df_hrv.to_csv(os.path.join(DATA_DIR, 'ecg_hrv_stats.csv'), index=False)
print("  Saved: data/ecg_hrv_stats.csv")

# =============================================================================
# 6. HRV DISTRIBUTION PLOTS
# =============================================================================
if HAS_SCIPY and len(df_hrv_valid) > 0:
    print("\n[6/8] Plotting HRV distributions...")
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle('HRV Metrics Distribution (187 Usable ECG Recordings)',
                 fontsize=13, fontweight='bold')

    metrics_plot = [
        ('peaks_count', 'Peak Count'),
        ('mean_rr',     'Mean RR (samples)'),
        ('sdnn',        'SDNN (samples)'),
        ('cv',          'CV (SDNN/Mean-RR)'),
    ]
    colors = ['steelblue', 'tomato', 'darkorange', 'mediumpurple']
    for ax, (col, lbl), color in zip(axes, metrics_plot, colors):
        data = df_hrv_valid[col].dropna()
        ax.hist(data, bins=25, color=color, alpha=0.85, edgecolor='white')
        ax.axvline(data.median(), color='black', linewidth=2, linestyle='--',
                   label=f'Median={data.median():.1f}')
        ax.set_title(lbl, fontsize=11, fontweight='bold')
        ax.set_xlabel(lbl)
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'ecg_hrv_distributions.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: reports/figures/ecg_hrv_distributions.png")
else:
    print("\n[6/8] Skipping HRV plots (no valid HRV data or scipy unavailable)")

# =============================================================================
# 7. ECG LABEL AMBIGUITY RESOLUTION  ← KEY DECISION
# =============================================================================
print("\n" + "=" * 70)
print("ECG LABEL AMBIGUITY RESOLUTION")
print("=" * 70)
print("""
FINDING 1 — NO GROUND-TRUTH LABELS:
  After thorough inspection, ecg_timeseries.csv contains NO label column.
  The 'Unnamed: 0' column is signal data (not patient IDs or labels).
  Column names are integer time-indices (deduplication artifacts create .1/.2/.3 variants).

FINDING 2 — ONLY 187/528 ROWS ARE USABLE:
  - Rows 0–186   (187 rows): Clean continuous ECG signals  ← ONLY THESE ARE USED
  - Rows 187–307 (121 rows): Sparse (mostly zero-padded)  ← EXCLUDED
  - Rows 308–527 (220 rows): Quantized (int-coded 1–4 scale) ← EXCLUDED

FINDING 3 — UNIFORMLY HIGH CV (CANNOT DERIVE CLEAN WEAK LABELS):
  All 187 usable rows show elevated Coefficient of Variation (CV > 0.20).
  This is caused by large inter-segment boundary gaps (the ECG signal is split
  across 5 non-contiguous column blocks), not by true clinical arrhythmia.
  Therefore, CV cannot reliably distinguish normal from abnormal rhythms.

MODELING DECISION — UNSUPERVISED ANOMALY DETECTION:
  Given the above, we adopt an UNSUPERVISED approach:
  
  → Model: 1D Convolutional Autoencoder (PyTorch)
  → Input: Windowed segments of 2000 samples (stride 1000, from 87,554-sample signal)
  → Output: Reconstruction error per window
  → Anomaly Score: Mean reconstruction error over all windows of a recording
  → High reconstruction error = potential rhythm irregularity / anomaly
  → This is a PROXY measure, NOT a clinical diagnosis
  
LIMITATIONS (explicitly documented per competition guidelines):
  1. No validated ground-truth labels exist for this dataset
  2. Only 35% (187/528) of recordings are usable
  3. Reconstruction error as an anomaly proxy has not been clinically validated
  4. Inter-segment boundary artifacts inflate HRV variability metrics
  5. The assumed 87,554-sample signal duration and sampling rate are inferred,
     not confirmed from dataset metadata
""")
print("=" * 70)

# =============================================================================
# 8. SAVE SIGNALS FOR NOTEBOOK 05
# =============================================================================
print("\n[8/8] Saving preprocessed ECG signals...")
np.save(os.path.join(DATA_DIR, 'ecg_signals_usable.npy'), signals)
print(f"  Saved: data/ecg_signals_usable.npy  (shape: {signals.shape})")

print("\n" + "=" * 70)
print("ECG EDA COMPLETE")
print("  Decision: Unsupervised 1D-CNN Autoencoder (reconstruction error = anomaly score)")
print("\n  Next: Notebook 05 — ECG Model (Autoencoder Training)")
print("=" * 70)


CardioScope — 04 ECG Time-Series EDA

[1/8] Structural inspection of ecg_timeseries.csv...


  Total columns:       123,995
  First 10 columns:    ['Unnamed: 0', '0', '1', '2', '3', '4', '5', '6', '7', '8']
  Last 10 columns:     ['4035.3', '4036.3', '4037.3', '4038.3', '4039.3', '4040.3', '4041.3', '4042.3', '4043.3', '4044.3']

  Column breakdown by naming pattern:
    Unnamed: 0 (index/signal col): 1
    Integer-named cols:            87553
    .1-suffix cols:                21891
    .2-suffix cols:                10505
    .3-suffix cols:                4045

  SIGNAL BLOCK STRUCTURE (discovered via boundary analysis):
    Block 1: cols index  1 to 21891 — 21,891 integer-named samples
    Block 2: cols index 21892 to 43782 — 21,891 .1-suffix samples
    Block 3: cols index 43783 to 109444 — 65,662 integer-named samples
    Block 4: cols index 109445 to 119949 — 10,505 .2-suffix samples
    Block 5: cols index 119950 to 123994 — 4,045 .3-suffix samples
    Unnamed: 0: first column — IS signal data (not a patient ID)

  USABLE SIGNAL = [Unnamed:0] + Block1 + Block3 = 87,554

  Loaded signals shape: (187, 87554)
  Value range: [0.0000, 1.0000]
  Mean ± Std:  0.1784 ± 0.2205

[3/8] Row subset classification:
  ┌──────────────┬────────────────────────────────────────────────────┐
  │ Rows 0–186   │ USABLE: Clean, continuous, normalized [0,1] signals │
  │ (187 rows)   │ → These are used for modeling                       │
  ├──────────────┼────────────────────────────────────────────────────┤
  │ Rows 187–307 │ EXCLUDED: Sparse signals (≥91% zero padding)        │
  │ (121 rows)   │ → Only isolated nonzero samples; no waveform shape  │
  ├──────────────┼────────────────────────────────────────────────────┤
  │ Rows 308–527 │ EXCLUDED: Quantized signals (discrete int scale 1–4) │
  │ (220 rows)   │ → Integer-coded, different amplitude scale           │
  └──────────────┴────────────────────────────────────────────────────┘

[4/8] Plotting sample waveforms...


  Saved: reports/figures/ecg_sample_waveforms.png

[5/8] Running peak detection and computing HRV metrics...


  Rows with valid HRV metrics: 187/187
  Mean RR interval: 230.0 samples
  Mean SDNN:        107.3 samples
  Mean CV:          0.4062
  CV range:         [0.2191, 2.4572]
  Saved: data/ecg_hrv_stats.csv

[6/8] Plotting HRV distributions...


  Saved: reports/figures/ecg_hrv_distributions.png

ECG LABEL AMBIGUITY RESOLUTION

FINDING 1 — NO GROUND-TRUTH LABELS:
  After thorough inspection, ecg_timeseries.csv contains NO label column.
  The 'Unnamed: 0' column is signal data (not patient IDs or labels).
  Column names are integer time-indices (deduplication artifacts create .1/.2/.3 variants).

FINDING 2 — ONLY 187/528 ROWS ARE USABLE:
  - Rows 0–186   (187 rows): Clean continuous ECG signals  ← ONLY THESE ARE USED
  - Rows 187–307 (121 rows): Sparse (mostly zero-padded)  ← EXCLUDED
  - Rows 308–527 (220 rows): Quantized (int-coded 1–4 scale) ← EXCLUDED

FINDING 3 — UNIFORMLY HIGH CV (CANNOT DERIVE CLEAN WEAK LABELS):
  All 187 usable rows show elevated Coefficient of Variation (CV > 0.20).
  This is caused by large inter-segment boundary gaps (the ECG signal is split
  across 5 non-contiguous column blocks), not by true clinical arrhythmia.
  Therefore, CV cannot reliably distinguish normal from abnormal rhythms.

MODELING D

  Saved: data/ecg_signals_usable.npy  (shape: (187, 87554))

ECG EDA COMPLETE
  Decision: Unsupervised 1D-CNN Autoencoder (reconstruction error = anomaly score)

  Next: Notebook 05 — ECG Model (Autoencoder Training)
